# Notatnik 3* ? Random Forest i hiperparametry

To jest notatnik z gwiazdk?. Hiperparametry s? wa?ne, ale ich strojenie kosztuje czas. W tym notebooku celowo uruchomimy ci??szy eksperyment, ?eby zobaczy? praktyczny koszt `GridSearchCV`.

B?dziemy trenowa? `RandomForestRegressor` na California Housing. Na typowym laptopie g??wny Grid Search mo?e potrwa? oko?o 2-4 minuty. Czas zale?y od procesora i liczby rdzeni.

## 1. Wczytanie danych i podzia?

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import pandas as pd
import time

housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Train full:", X_train_full.shape)
print("Test:", X_test.shape)

## 2. Dlaczego u?yjemy podzbioru treningowego?

Pe?ny Grid Search na ca?ym zbiorze mo?e trwa? d?ugo. W warsztacie chcemy pokaza? mechanik? i koszt strojenia, wi?c stroimy na kontrolowanym podzbiorze, a finalnie oceniamy najlepszy model na pe?nym te?cie.

In [ ]:
X_tune, _, y_tune, _ = train_test_split(
    X_train_full,
    y_train_full,
    train_size=10_000,
    random_state=42,
)

print("Dane u?yte do strojenia:", X_tune.shape)

## 3. Model bazowy Random Forest

Zanim uruchomimy Grid Search, trenujemy rozs?dny model bazowy. Dzi?ki temu wiemy, czy strojenie realnie co? poprawia.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate(name, model, X_train, y_train, X_test, y_test):
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    return {
        "model": name,
        "train_MAE": mean_absolute_error(y_train, pred_train),
        "test_MAE": mean_absolute_error(y_test, pred_test),
        "test_RMSE": mean_squared_error(y_test, pred_test) ** 0.5,
        "test_R2": r2_score(y_test, pred_test),
    }

start = time.perf_counter()
base_forest = RandomForestRegressor(
    n_estimators=150,
    random_state=42,
    n_jobs=-1,
)
print("Trenuj? model bazowy Random Forest...")
base_forest.fit(X_train_full, y_train_full)
base_seconds = time.perf_counter() - start
print(f"Model bazowy gotowy po {base_seconds:.1f} s")

base_result = evaluate("RandomForest baseline", base_forest, X_train_full, y_train_full, X_test, y_test)
base_result

## 4. Siatka hiperparametr?w

Hiperparametry to ustawienia modelu wybrane przed treningiem. Dla Random Forest przyk?adowe wa?ne hiperparametry to:

- `n_estimators` ? liczba drzew,
- `max_depth` ? maksymalna g??boko?? drzewa,
- `min_samples_leaf` ? minimalna liczba przyk?ad?w w li?ciu,
- `max_features` ? ile cech ka?de drzewo mo?e rozwa?a? przy podziale.

In [ ]:
param_grid = {
    "n_estimators": [150, 250, 350],
    "max_depth": [None, 12, 24],
    "min_samples_leaf": [1, 3, 8],
    "max_features": [0.6, 0.8, 1.0],
}

num_candidates = 1
for values in param_grid.values():
    num_candidates *= len(values)

cv = 3
total_fits = num_candidates * cv

print("Liczba kombinacji hiperparametr?w:", num_candidates)
print("Liczba fold?w CV:", cv)
print("??czna liczba trening?w:", total_fits)
print("\nUwaga: ten krok celowo mo?e potrwa? oko?o 2-4 minuty na typowym laptopie.")

## 5. GridSearchCV z widocznym post?pem

`verbose=2` wypisuje post?p kolejnych dopasowa?. To przydatne, gdy eksperyment trwa d?u?ej ni? kilka sekund.

In [ ]:
from sklearn.model_selection import GridSearchCV

forest = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
)

grid_search = GridSearchCV(
    estimator=forest,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
)

print("Start GridSearchCV...")
print(f"Do wykonania: {total_fits} trening?w. To mo?e potrwa? kilka minut.")
start = time.perf_counter()

grid_search.fit(X_tune, y_tune)

grid_seconds = time.perf_counter() - start
print(f"Grid Search zako?czony po {grid_seconds / 60:.1f} min ({grid_seconds:.1f} s).")

## 6. Najlepsze parametry i wynik CV

In [ ]:
best_cv_mae = -grid_search.best_score_

print("Najlepsze parametry:")
for key, value in grid_search.best_params_.items():
    print(f"- {key}: {value}")

print(f"\nNajlepsze CV MAE: {best_cv_mae:.4f}")
print(f"Czyli oko?o {best_cv_mae * 100_000:,.0f} USD przeci?tnego b??du.")

## 7. Finalny model na pe?nym zbiorze treningowym

Grid Search szuka? parametr?w na podzbiorze danych, ?eby warsztat dzia?a? w rozs?dnym czasie. Po wyborze parametr?w trenujemy finalny model na pe?nym zbiorze treningowym i dopiero ten model oceniamy na te?cie.


In [ ]:
print("Trenuj? finalny Random Forest na pe?nym zbiorze treningowym...")
start = time.perf_counter()

final_model = RandomForestRegressor(
    **grid_search.best_params_,
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X_train_full, y_train_full)

final_seconds = time.perf_counter() - start
print(f"Finalny model gotowy po {final_seconds:.1f} s")

final_result = evaluate("RandomForest GridSearch + full train", final_model, X_train_full, y_train_full, X_test, y_test)
comparison = pd.DataFrame([base_result, final_result])
comparison["test_MAE_USD"] = comparison["test_MAE"].map(lambda x: f"{x * 100_000:,.0f} USD")
comparison["test_RMSE_USD"] = comparison["test_RMSE"].map(lambda x: f"{x * 100_000:,.0f} USD")
comparison


## 8. Top konfiguracje

Wyniki Grid Search warto przegl?da?. Czasem kilka konfiguracji ma prawie taki sam wynik, wi?c nie warto udawa?, ?e jedna liczba po przecinku oznacza wielk? r??nic?.

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
columns = [
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_leaf",
    "param_max_features",
    "mean_test_score",
    "mean_train_score",
    "mean_fit_time",
]

results_view = results[columns].copy()
results_view["CV_MAE"] = -results_view["mean_test_score"]
results_view["TRAIN_MAE"] = -results_view["mean_train_score"]

results_view.sort_values("CV_MAE").head(15)

## 9. Overfitting w wynikach Grid Search

Je?li `TRAIN_MAE` jest du?o ni?sze ni? `CV_MAE`, model ?wietnie pasuje do danych treningowych, ale s?abiej generalizuje. To jeden z praktycznych sygna??w overfittingu.

In [ ]:
results_view["gap_train_cv"] = results_view["CV_MAE"] - results_view["TRAIN_MAE"]

results_view.sort_values("gap_train_cv", ascending=False).head(10)

## 10. Wa?no?? cech najlepszego modelu

In [ ]:
import matplotlib.pyplot as plt

importance = pd.Series(final_model.feature_importances_, index=X.columns).sort_values()
importance.plot(kind="barh", title="Wa?no?? cech ? najlepszy Random Forest")
plt.xlabel("feature_importance")
plt.show()

importance.sort_values(ascending=False)

## 11. Zapis modelu

W praktyce cz?sto zapisujemy wytrenowany model, ?eby u?y? go p??niej bez ponownego treningu.

In [ ]:
from pathlib import Path
import joblib

output_dir = Path("models")
output_dir.mkdir(exist_ok=True)
model_path = output_dir / "random_forest_california.joblib"

joblib.dump(final_model, model_path)
print("Model zapisany do:", model_path)

loaded_model = joblib.load(model_path)
print("Model wczytany. Przyk?adowa predykcja:", loaded_model.predict(X_test.iloc[[0]])[0])

## Twoja kolej

1. Zmniejsz siatk? parametr?w i sprawd?, ile czasu oszcz?dzasz.
2. Zwi?ksz `n_estimators` i zobacz, czy wynik poprawia si? proporcjonalnie do czasu.
3. Por?wnaj najlepszy model z baseline Random Forest. Czy strojenie by?o warte czasu?

## Podsumowanie

Najwa?niejsze lekcje:

- hiperparametry realnie wp?ywaj? na model,
- Grid Search jest prosty w u?yciu, ale szybko robi si? kosztowny,
- `verbose` i liczenie liczby trening?w pomagaj? planowa? eksperyment,
- najlepszy wynik CV trzeba sprawdzi? na zbiorze testowym,
- strojenie hiperparametr?w nie zast?puje my?lenia o danych, metryce i koszcie oblicze?.